# Deliverable 3: Leakage Analysis

This notebook evaluates forecasting-table columns for predicting OD_PAX for an origin, destination, and departure month. The goal is to use only information available before departure and avoid target or post-departure leakage.

In [ ]:
from pathlib import Path
import pandas as pd

# Find the project root from the notebook kernel's current directory.
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'data' / 'raw' / 'AA_PAX_FORECASTING.csv').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not find data/raw/AA_PAX_FORECASTING.csv from the current folder.')

data_dir = project_root / 'data' / 'raw'
airport_df = pd.read_csv(data_dir / 'AA_AIRPORT_INFO.csv')
forecast_df = pd.read_csv(data_dir / 'AA_PAX_FORECASTING.csv')

In [ ]:
print(f'Airport information table: {airport_df.shape[0]:,} rows, {airport_df.shape[1]:,} columns')
print(f'Forecasting table: {forecast_df.shape[0]:,} rows, {forecast_df.shape[1]:,} columns')

## Shape

The forecasting table contains 1,533,522 rows and 8 columns, with one row per origin, destination, and departure month. The airport information table contains 4,565 rows and 18 columns.

In [ ]:
forecast_df.dtypes

## Leakage review

The prediction is for a specified origin, destination, and departure month. Fields whose meaning or timing could not be confirmed are excluded provisionally rather than treated as safe predictors.

| Column | Known before departure? | Keep or drop? | Reason |
|---|---|---|---|
| Unnamed: 0 | Unconfirmed; PM clarification pending | Drop provisionally | Its meaning and timing are undocumented, so it is excluded until confirmed. |
| DEP_MONTH | Yes | Keep | The requested departure month is known when the forecast is made. |
| DEP_YEAR | Yes | Keep | The requested departure year is known when the forecast is made. |
| ORG_AIRPRT | Yes | Keep | The origin airport is specified for the route being forecast. |
| DES_AIRPRT | Yes | Keep | The destination airport is specified for the route being forecast. |
| OD_PAX | No; this is the outcome | Drop from predictors; retain as target | The actual passenger count is what the model predicts and is unavailable as an input at prediction time. |
| OD_REV_USD | Unconfirmed; PM clarification pending | Drop provisionally | Its definition and pre-departure availability are unconfirmed, so including it could leak post-departure information. |
| SF_LOAD_TMS | Unconfirmed; PM clarification pending | Drop | It has one unique value in the file, so it provides no variation for a model; its timing remains unconfirmed. |

In [ ]:
missing_count = forecast_df.isna().sum()
missing_percent = forecast_df.isna().mean() * 100

pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

## Missingness

The forecasting table has no missing values: all 8 columns have 0 missing entries (0.0%).

## Distributions

The following summaries use the full forecasting table; the histogram below uses the stated reproducible sample.

In [ ]:
forecast_df["OD_PAX"].describe()


In [ ]:
forecast_df["OD_PAX"].skew()

In [ ]:
import matplotlib.pyplot as plt

forecast_sample = forecast_df.sample(n=50_000, random_state=9)

plt.hist(forecast_sample["OD_PAX"], bins=50)
plt.title("OD_PAX Distribution (50,000-row sample, random_state=9)")
plt.xlabel("OD_PAX")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

OD_PAX is strongly right-skewed: the full-data skewness is about 5.32, and the histogram shows most sampled values near the lower end with a long tail toward higher passenger counts. The histogram uses a 50,000-row sample selected with random_state=9.

In [ ]:
forecast_df["SF_LOAD_TMS"].nunique(dropna=False)

SF_LOAD_TMS is excluded as a predictor because it has only one unique value across the forecasting table and therefore provides no variation for the model. This does not establish when the field becomes available; its definition and timing remain pending PM clarification.

## Model choice

Gradient-boosted trees are a reasonable choice for this structured route-and-month data because they can learn nonlinear relationships and interactions among tabular features. The target OD_PAX is kept separate from the input features to prevent direct target leakage; OD_REV_USD and Unnamed: 0 remain excluded until their meanings and pre-departure availability are confirmed.